# JRT results: graded response model, model ladder and reliability

Display notebook for the manuscript. It fits nothing: it shows the CSV and figure
outputs of the R pipeline in `../R/` (`01_data_design.R` … `06_summary.R`, run on the
server, see `../server/README.md` and `../AUDIT.md`) as labelled tables and figures,
so the manuscript can embed them. Re-execute it (seconds) after every pipeline run.
Objects whose source file does not exist yet show a placeholder line.

Numbers in the text come from these tables through the results-interpreter.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Markdown

RES = Path("..").resolve() / "results"

def _missing(path):
    return Markdown(f"_Not available yet: `results/{path}` (run the R pipeline)._")

def read_res(path):
    f = RES / path
    return pd.read_csv(f) if f.exists() else None

def show_table(path, fn=None, digits=3):
    # Markdown table without the index column, for embedding in the manuscript.
    df = read_res(path)
    if df is None:
        return _missing(path)
    if fn is not None:
        df = fn(df)
    return Markdown(df.to_markdown(index=False, floatfmt=f".{digits}f"))

def show_png(path, width=8):
    f = RES / path
    if not f.exists():
        return _missing(path)
    img = mpimg.imread(f)
    fig, ax = plt.subplots(figsize=(width, width * img.shape[0] / img.shape[1]))
    ax.imshow(img)
    ax.axis("off")
    fig.tight_layout(pad=0)
    plt.show()

## 1. Data and design

In [2]:
#| label: tbl-design-summary
#| tbl-cap: "Data basis shared by all models: ratings, products, raters, coverage of the product x rater matrix and connectivity of the design."
show_table("data/design_summary.csv", lambda d: d[d["quantity"] != "data_fingerprint"])

| quantity                          | value              |
|:----------------------------------|:-------------------|
| n_ratings                         | 7844               |
| n_products                        | 3669               |
| n_raters                          | 22                 |
| n_categories                      | 7                  |
| scale                             | 1-7                |
| internal_coding                   | rating - 1 (0-6)   |
| ratings_per_product_mean          | 2.13791223766694   |
| ratings_per_product_median        | 2                  |
| ratings_per_product_min           | 1                  |
| ratings_per_product_max           | 12                 |
| ratings_per_product_harmonic_mean | 1.72632034132518   |
| products_with_1_rating            | 898                |
| ratings_per_rater_min             | 2                  |
| ratings_per_rater_max             | 1174               |
| matrix_fill                       | 0.0971778289848609 |
| n_components                      | 1                  |
| rater_pairs_total                 | 231                |
| rater_pairs_without_overlap       | 37                 |
| rater_pairs_overlap_median        | 7                  |
| raters_with_category_gaps         | 8                  |
| raters_with_internal_gap          | 2                  |
| duplicate_pairs_resolved_in_00    | 12                 |
| duplicate_pairs_conflicting       | 1                  |

In [3]:
#| label: tbl-category-usage
#| tbl-cap: "Category use per rater (counts of ratings 1-7). Unused categories at the ends of the scale reduce the number of thresholds of a rater; an unused category inside the observed range (internal gap) makes that rater's thresholds incomparable with those of other raters."
show_table("data/category_usage.csv", lambda d: d[["rater_id"] + [c for c in d if c.startswith("cat_")] + ["n_ratings", "missing_categories", "internal_gap"]])

| rater_id   |   cat_1 |   cat_2 |   cat_3 |   cat_4 |   cat_5 |   cat_6 |   cat_7 |   n_ratings | missing_categories   | internal_gap   |
|:-----------|--------:|--------:|--------:|--------:|--------:|--------:|--------:|------------:|:---------------------|:---------------|
| R01        |     337 |     461 |     259 |      87 |      24 |       6 |       0 |        1174 | 7                    | False          |
| R02        |     241 |     179 |     124 |     126 |      91 |      67 |      45 |         873 | nan                  | False          |
| R03        |     161 |     182 |     185 |      82 |      88 |      22 |       0 |         720 | 7                    | False          |
| R04        |     164 |     164 |      90 |      69 |      92 |      36 |       5 |         620 | nan                  | False          |
| R05        |     159 |     190 |     102 |      90 |      50 |      30 |      12 |         633 | nan                  | False          |
| R06        |     301 |     113 |      57 |      34 |       9 |       6 |       0 |         520 | 7                    | False          |
| R07        |     183 |     152 |      96 |      61 |      36 |      17 |       2 |         547 | nan                  | False          |
| R08        |     101 |     127 |     106 |      91 |      60 |      30 |       7 |         522 | nan                  | False          |
| R09        |      63 |      62 |      72 |      60 |      66 |      55 |      22 |         400 | nan                  | False          |
| R10        |      25 |     156 |      76 |      64 |      47 |      13 |       2 |         383 | nan                  | False          |
| R11        |     127 |     118 |      51 |      31 |      32 |       9 |       1 |         369 | nan                  | False          |
| R12        |      11 |      48 |      90 |      77 |      27 |      10 |       1 |         264 | nan                  | False          |
| R13        |       4 |      35 |      68 |      91 |      44 |      11 |       1 |         254 | nan                  | False          |
| R14        |     105 |      66 |      40 |      31 |      10 |       5 |       2 |         259 | nan                  | False          |
| R15        |       7 |      22 |      11 |       9 |      13 |       9 |       2 |          73 | nan                  | False          |
| R16        |       4 |      14 |       9 |      14 |       5 |       3 |       3 |          52 | nan                  | False          |
| R17        |      25 |      10 |       9 |       4 |       4 |       0 |       0 |          52 | 6;7                  | False          |
| R18        |       1 |       8 |      14 |      12 |      16 |       4 |       0 |          55 | 7                    | False          |
| R19        |      10 |       6 |       8 |       6 |      10 |       6 |       4 |          50 | nan                  | False          |
| R20        |       4 |       0 |       1 |       4 |       2 |       2 |       0 |          13 | 2;7                  | True           |
| R21        |       1 |       2 |       3 |       1 |       2 |       0 |       0 |           9 | 6;7                  | False          |
| R22        |       0 |       1 |       0 |       1 |       0 |       0 |       0 |           2 | 1;3;5;6;7            | True           |

## 2. Graded response model (central model)

In [4]:
#| label: tbl-grm-convergence
#| tbl-cap: "Estimation and convergence of the graded response model (marginal maximum likelihood, EM algorithm, fixed quadrature)."
show_table("grm/convergence.csv", lambda d: d[["method", "quadpts", "se_type", "tol", "converged", "iterations", "information_matrix_pd", "logLik", "npar", "AIC", "BIC", "SABIC", "n_products", "n_ratings"]])

_Not available yet: `results/grm/convergence.csv` (run the R pipeline)._

In [5]:
#| label: tbl-grm-rater-parameters
#| tbl-cap: "Rater parameters of the graded response model in IRT metric: discrimination a and thresholds b_k with standard errors; category_boundaries states which rating categories each threshold separates."
show_table("grm/rater_parameters.csv", lambda d: d[["rater_id", "n_ratings", "a", "a_se"] + [c for c in d if c[0] == "b" and c[1:].isdigit()] + ["category_boundaries", "thresholds_ordered"]])

_Not available yet: `results/grm/rater_parameters.csv` (run the R pipeline)._

In [6]:
#| label: fig-grm-category-response-functions
#| fig-cap: "Category response functions of the graded response model for each rater."
show_png("grm/figures/grm_category_response_functions.png", width=11)

_Not available yet: `results/grm/figures/grm_category_response_functions.png` (run the R pipeline)._

In [7]:
#| label: tbl-grm-rater-fit
#| tbl-cap: "Rater fit in the graded response model: infit and outfit mean squares (observed ratings) and S-X2 pooled over multiple imputations of the missing ratings."
show_table("grm/rater_fit.csv", lambda d: d[["rater_id", "n_ratings", "infit", "outfit", "S_X2", "df_S_X2", "p_S_X2", "n_imputations"]])

_Not available yet: `results/grm/rater_fit.csv` (run the R pipeline)._

In [8]:
#| label: tbl-grm-q3-flagged
#| tbl-cap: "Rater pairs with the largest local dependence (Q3 minus mean Q3) among pairs with at least 30 jointly rated products."
show_table("grm/q3.csv", lambda d: d[d["n_common"] >= 30].assign(_a=lambda x: x["q3_minus_mean"].abs()).sort_values("_a", ascending=False).drop(columns="_a").head(10))

_Not available yet: `results/grm/q3.csv` (run the R pipeline)._

In [9]:
#| label: tbl-grm-global-fit
#| tbl-cap: "Limited-information absolute fit of the graded response model (C2)."
show_table("grm/global_fit.csv", lambda d: d.drop(columns=["q3_mirt_crosscheck_max_abs_diff"], errors="ignore"))

_Not available yet: `results/grm/global_fit.csv` (run the R pipeline)._

## 3. Information and reliability

In [10]:
#| label: fig-grm-rater-information
#| fig-cap: "Rater information functions of the graded response model."
show_png("grm/figures/grm_rater_information.png")

_Not available yet: `results/grm/figures/grm_rater_information.png` (run the R pipeline)._

In [11]:
#| label: fig-grm-information-shares
#| fig-cap: "Share of each rater in the information of the full panel across the latent continuum."
show_png("grm/figures/grm_rater_information_shares.png")

_Not available yet: `results/grm/figures/grm_rater_information_shares.png` (run the R pipeline)._

In [12]:
#| label: tbl-grm-information-summary
#| tbl-cap: "Rater information summary: information integrated over N(0, 1), location and height of the maximum, and share of the full-panel information (weighted over N(0, 1) and at selected theta)."
show_table("grm/rater_information_summary.csv")

_Not available yet: `results/grm/rater_information_summary.csv` (run the R pipeline)._

In [13]:
#| label: fig-grm-panel-information
#| fig-cap: "Information and conditional reliability I/(I+1) of the full panel. The curve is hypothetical: it applies to a product rated by all raters."
show_png("grm/figures/grm_panel_information_reliability.png")

_Not available yet: `results/grm/figures/grm_panel_information_reliability.png` (run the R pipeline)._

In [14]:
#| label: tbl-grm-reliability-summary
#| tbl-cap: "Reliability of the GRM scores: empirical marginal reliability of the actual EAP scores (primary), model-based marginal reliability of the hypothetical full panel, distribution of the product-level reliability rel_post = 1 - PSD^2, and its agreement with the information-based rel_info."
show_table("grm/reliability_summary.csv")

_Not available yet: `results/grm/reliability_summary.csv` (run the R pipeline)._

In [15]:
#| label: tbl-grm-reliability-by-n
#| tbl-cap: "Product-level reliability and posterior SD by number of ratings per product."
show_table("grm/reliability_by_n_ratings.csv")

_Not available yet: `results/grm/reliability_by_n_ratings.csv` (run the R pipeline)._

In [16]:
#| label: fig-grm-rel-post-histogram
#| fig-cap: "Distribution of the product-level reliability rel_post across all products."
show_png("grm/figures/grm_rel_post_histogram.png")

_Not available yet: `results/grm/figures/grm_rel_post_histogram.png` (run the R pipeline)._

In [17]:
#| label: fig-grm-rel-post-by-n
#| fig-cap: "Product-level reliability rel_post by number of ratings per product."
show_png("grm/figures/grm_rel_post_by_n_ratings.png")

_Not available yet: `results/grm/figures/grm_rel_post_by_n_ratings.png` (run the R pipeline)._

In [18]:
#| label: fig-grm-rel-post-by-theta
#| fig-cap: "Product-level reliability rel_post by EAP estimate."
show_png("grm/figures/grm_rel_post_by_theta.png")

_Not available yet: `results/grm/figures/grm_rel_post_by_theta.png` (run the R pipeline)._

In [19]:
#| label: fig-grm-rel-consistency
#| fig-cap: "Agreement of the two product-level reliability indices: rel_info (information of the product's own raters at the EAP estimate) and rel_post (1 - PSD^2)."
show_png("grm/figures/grm_rel_consistency.png", width=6)

_Not available yet: `results/grm/figures/grm_rel_consistency.png` (run the R pipeline)._

## 4. Comparison: model ladder and MFRM

In [20]:
#| label: tbl-model-fit
#| tbl-cap: "Fit of the adjacent-category ladder (M1 rating scale model / MFRM, M2 partial credit model, M3 generalized partial credit model) and the graded response model: log-likelihood, number of parameters and information criteria."
show_table("comparison/model_fit.csv", lambda d: d[d["status"] == "estimated"][["model", "software", "family", "logLik", "npar", "AIC", "BIC", "SABIC"]])

_Not available yet: `results/comparison/model_fit.csv` (run the R pipeline)._

In [21]:
#| label: tbl-lr-tests
#| tbl-cap: "Likelihood-ratio tests within the adjacent-category ladder."
show_table("comparison/lr_tests.csv", lambda d: d[["comparison", "X2", "df", "df_expected_formula", "p"]])

_Not available yet: `results/comparison/lr_tests.csv` (run the R pipeline)._

In [22]:
#| label: tbl-theta-correlations
#| tbl-cap: "Correlations of the EAP estimates between models (Pearson and Spearman)."
show_table("comparison/theta_correlations.csv")

_Not available yet: `results/comparison/theta_correlations.csv` (run the R pipeline)._

In [23]:
#| label: tbl-discrimination-m3
#| tbl-cap: "Rater discriminations in the generalized partial credit model (M3)."
show_table("comparison/discrimination_m3.csv", lambda d: d[["rater_id", "n_ratings", "estimate", "se"]])

_Not available yet: `results/comparison/discrimination_m3.csv` (run the R pipeline)._

In [24]:
#| label: tbl-threshold-order
#| tbl-cap: "Raters with unordered step parameters in M2 (PCM) and M3 (GPCM)."
show_table("comparison/threshold_order.csv", lambda d: d[~d["ordered"]])

_Not available yet: `results/comparison/threshold_order.csv` (run the R pipeline)._

In [25]:
#| label: tbl-rater-fit-m1
#| tbl-cap: "Rater infit and outfit mean squares under M1 (rating scale model / MFRM)."
show_table("comparison/rater_fit_m1.csv", lambda d: d[["rater_id", "n_ratings", "infit", "outfit"]])

_Not available yet: `results/comparison/rater_fit_m1.csv` (run the R pipeline)._

In [26]:
#| label: tbl-marginal-reliability-models
#| tbl-cap: "Marginal and empirical reliability by model, for comparison."
show_table("comparison/marginal_reliability.csv")

_Not available yet: `results/comparison/marginal_reliability.csv` (run the R pipeline)._

## 5. Naive indices and overview

In [27]:
def naive_indices():
    cr, ka = read_res("naive/cronbach_alpha.csv"), read_res("naive/krippendorff_alpha.csv")
    if cr is None or ka is None:
        return None
    return pd.concat([
        pd.DataFrame({"index": "Cronbach's alpha: " + cr["variant"], "value": cr["raw_alpha"],
                      "ci_low": None, "ci_high": None, "n_raters": cr["n_raters"],
                      "n_products": cr["n_products_contributing"]}),
        pd.DataFrame({"index": "Krippendorff's alpha (ordinal)", "value": ka["alpha"],
                      "ci_low": ka["ci_low"], "ci_high": ka["ci_high"], "n_raters": ka["n_raters"],
                      "n_products": ka["n_pairable_units"]}),
    ], ignore_index=True)

In [28]:
#| label: tbl-naive-indices
#| tbl-cap: "Cronbach's alpha (consistency of the unweighted rater mean) and Krippendorff's alpha (ordinal agreement of single ratings, 95% bootstrap CI over products)."
_naive = naive_indices()
_missing("naive/*.csv") if _naive is None else Markdown(_naive.to_markdown(index=False, floatfmt=".3f"))

_Not available yet: `results/naive/*.csv` (run the R pipeline)._

In [29]:
#| label: tbl-reliability-overview
#| tbl-cap: "Overview of the reliability and agreement indices and what each refers to."
show_table("summary/reliability_overview.csv")

_Not available yet: `results/summary/reliability_overview.csv` (run the R pipeline)._